# Prediction report analysis

Notebook for analyzing files from the `reports/` folder. Compares model accuracy across a selected date range.

**Requirements:**
- Only considers reports with `finished` status
- Skips postponed matches and those without an official result
- User selects date range (default: all available)

In [ ]:
import json
import os
import glob
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

REPORTS_DIR = 'reports'
print(f"Reports folder: {os.path.abspath(REPORTS_DIR)}")

date_dirs = [d for d in sorted(os.listdir(REPORTS_DIR))
             if os.path.isdir(os.path.join(REPORTS_DIR, d)) and len(d) == 10 and d[4] == '-']
print(f"Available dates: {len(date_dirs)}")

In [ ]:
# ============================================================
# CONFIGURATION - Set date range
# ============================================================
# None = no limit (use all available)
# Format: 'YYYY-MM-DD'
DATE_FROM = None       # e.g. '2026-02-08'
DATE_TO   = None       # e.g. '2026-02-10'
# ============================================================

def load_reports(reports_dir, date_from=None, date_to=None):
    """Load reports from folder, filter by date range and status.
    
    Expected structure: reports/YYYY-MM-DD/predictions_finished.json
    """
    reports = []
    skipped = []
    
    for date_dir in sorted(os.listdir(reports_dir)):
        date_path = os.path.join(reports_dir, date_dir)
        if not os.path.isdir(date_path) or len(date_dir) != 10 or date_dir[4] != '-':
            continue
        
        if date_from and date_dir < date_from:
            continue
        if date_to and date_dir > date_to:
            continue
        
        pred_file = None
        for status in ['finished', 'unfinished']:
            path = os.path.join(date_path, f'predictions_{status}.json')
            if os.path.exists(path):
                pred_file = path
                break
        
        if not pred_file:
            skipped.append((date_dir, 'brak pliku predictions'))
            continue
        
        try:
            with open(pred_file, 'r', encoding='utf-8') as f:
                report = json.load(f)
        except Exception as e:
            skipped.append((date_dir, f'read error: {e}'))
            continue
        
        report_status = report.get('status', '')
        if report_status != 'finished':
            skipped.append((date_dir, f'status={report_status}'))
            continue
        
        report['_date'] = date_dir
        reports.append(report)
    
    return reports, skipped

reports, skipped = load_reports(REPORTS_DIR, DATE_FROM, DATE_TO)

print(f"Loaded reports: {len(reports)}")
if skipped:
    print(f"Skipped: {len(skipped)}")
    for d, reason in skipped:
        print(f"  - {d}: {reason}")

if reports:
    dates = [r['_date'] for r in reports]
    print(f"\nRange: {min(dates)} -> {max(dates)}")
    for r in reports:
        s = r.get('summary', {})
        print(f"  {r['_date']}: {s.get('finished_matches', '?')} matches "
              f"(+{s.get('postponed_matches', 0)} postponed)")
else:
    print("\nNo reports to analyze! Check date range or reports/ folder")

In [ ]:
rows = []
for report in reports:
    report_date = report.get('_date', report.get('date', ''))
    for match in report.get('matches', []):
        # Skip postponed and without result
        if match.get('status') != 'finished':
            continue
        if match.get('actual_result') is None:
            continue
        
        actual = match['actual_result']
        league = match.get('league', 'unknown')
        score = match.get('actual_score', '')
        home = match.get('home_team', '')
        away = match.get('away_team', '')
        consensus = match.get('consensus', {})
        
        for model_name, pred_data in match.get('predictions', {}).items():
            prediction = pred_data.get('prediction')
            confidence = pred_data.get('confidence', 0)
            correct = pred_data.get('correct')
            probs = pred_data.get('probabilities', {})
            
            rows.append({
                'date': report_date,
                'league': league,
                'home_team': home,
                'away_team': away,
                'actual_result': actual,
                'actual_score': score,
                'model': model_name,
                'prediction': prediction,
                'confidence': confidence,
                'correct': correct,
                'prob_home': probs.get('HOME', 0),
                'prob_draw': probs.get('DRAW', 0),
                'prob_away': probs.get('AWAY', 0),
                'consensus_pred': consensus.get('prediction'),
                'consensus_agreement': consensus.get('agreement_pct', 0),
                'consensus_correct': consensus.get('correct'),
            })

df = pd.DataFrame(rows)

if len(df) == 0:
    raise ValueError("Brak danych - zaden mecz nie spelnia kryteriow!")

n_matches = df.groupby(['date', 'home_team', 'away_team']).ngroups
n_models = df['model'].nunique()
n_dates = df['date'].nunique()
n_leagues = df['league'].nunique()

print(f"Loaded: {len(df)} predictions")
print(f"  Unique matches: {n_matches}")
print(f"  Models: {n_models} ({', '.join(sorted(df['model'].unique()))})")
print(f"  Days: {n_dates}")
print(f"  Leagues: {n_leagues}")

## 1. Overall model accuracy

In [ ]:
model_stats = df.groupby('model').agg(
    total=('correct', 'count'),
    correct=('correct', 'sum'),
    avg_confidence=('confidence', 'mean'),
).reset_index()
model_stats['accuracy'] = model_stats['correct'] / model_stats['total']
model_stats = model_stats.sort_values('accuracy', ascending=False).reset_index(drop=True)
model_stats.index += 1  # ranking from 1

display_df = model_stats.copy()
display_df['accuracy'] = display_df['accuracy'].map(lambda x: f"{x:.1%}")
display_df['avg_confidence'] = display_df['avg_confidence'].map(lambda x: f"{x:.1f}%")
display_df.columns = ['Model', 'Matches', 'Correct', 'Avg. confidence', 'Accuracy']
print(f"=== Accuracy models ({n_matches} matches, {n_dates} days) ===\n")
display(display_df)

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(model_stats)))
bars = ax.barh(model_stats['model'], model_stats['accuracy'], color=colors, edgecolor='black')
for bar, acc in zip(bars, model_stats['accuracy']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{acc:.1%}', va='center', fontsize=11, fontweight='bold')
ax.set_xlabel('Accuracy', fontsize=12)
ax.set_title(f'Accuracy models - {min(df["date"])} -> {max(df["date"])} ({n_matches} matches)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, model_stats['accuracy'].max() * 1.15)
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 2. Accuracy per day (trend)

In [ ]:
daily = df.groupby(['date', 'model']).agg(
    correct=('correct', 'sum'),
    total=('correct', 'count'),
).reset_index()
daily['accuracy'] = daily['correct'] / daily['total']

pivot = daily.pivot(index='date', columns='model', values='accuracy')

fig, ax = plt.subplots(figsize=(14, 6))
markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*']
for i, model in enumerate(pivot.columns):
    ax.plot(pivot.index, pivot[model], marker=markers[i % len(markers)],
            linewidth=2, markersize=7, label=model)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Model accuracy per day', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

daily_table = pivot.copy()
for col in daily_table.columns:
    daily_table[col] = daily_table[col].map(lambda x: f"{x:.1%}" if pd.notna(x) else '-')
matches_per_day = df.groupby('date').apply(
    lambda g: g.groupby(['home_team', 'away_team']).ngroups
)
daily_table.insert(0, 'Matches', matches_per_day)
print("=== Accuracy per day ===")
display(daily_table)

## 3. Accuracy per league

In [ ]:
league_model = df.groupby(['league', 'model']).agg(
    correct=('correct', 'sum'),
    total=('correct', 'count'),
).reset_index()
league_model['accuracy'] = league_model['correct'] / league_model['total']

league_pivot = league_model.pivot(index='league', columns='model', values='accuracy')
league_counts = df.groupby('league').apply(
    lambda g: g.groupby(['home_team', 'away_team']).ngroups
).sort_values(ascending=False)

valid_leagues = league_counts[league_counts >= 2].index
league_pivot = league_pivot.loc[league_pivot.index.isin(valid_leagues)]
league_pivot = league_pivot.loc[league_counts[valid_leagues].index]

fig, ax = plt.subplots(figsize=(14, max(5, len(league_pivot) * 0.5)))
im = ax.imshow(league_pivot.values, cmap='RdYlGn', aspect='auto', vmin=0.2, vmax=0.8)

ax.set_xticks(range(len(league_pivot.columns)))
ax.set_xticklabels(league_pivot.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(league_pivot.index)))
league_labels = [f"{l} ({league_counts.get(l, 0)})" for l in league_pivot.index]
ax.set_yticklabels(league_labels, fontsize=9)

for i in range(len(league_pivot.index)):
    for j in range(len(league_pivot.columns)):
        val = league_pivot.iloc[i, j]
        if pd.notna(val):
            ax.text(j, i, f'{val:.0%}', ha='center', va='center', fontsize=8,
                    color='black' if 0.35 < val < 0.65 else 'white')

plt.colorbar(im, ax=ax, label='Accuracy')
ax.set_title('Accuracy per league and model (leagues with ≥2 matches)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== Best model per league ===")
best_per_league = league_model.loc[league_model.groupby('league')['accuracy'].idxmax()]
best_per_league = best_per_league.sort_values('accuracy', ascending=False)
for _, row in best_per_league.iterrows():
    print(f"  {row['league']:35s} -> {row['model']:20s} ({row['accuracy']:.1%}, {int(row['total'])} matches)")

## 4. Accuracy per result type (H/D/A)

In [ ]:
result_stats = df.groupby(['model', 'actual_result']).agg(
    correct=('correct', 'sum'),
    total=('correct', 'count'),
).reset_index()
result_stats['accuracy'] = result_stats['correct'] / result_stats['total']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
result_order = ['HOME', 'DRAW', 'AWAY']
result_colors = {'HOME': '#27ae60', 'DRAW': '#f39c12', 'AWAY': '#e74c3c'}

for ax, result_type in zip(axes, result_order):
    subset = result_stats[result_stats['actual_result'] == result_type].sort_values('accuracy', ascending=True)
    bars = ax.barh(subset['model'], subset['accuracy'],
                   color=result_colors[result_type], edgecolor='black', alpha=0.85)
    for bar, (_, row) in zip(bars, subset.iterrows()):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f"{row['accuracy']:.0%} ({int(row['correct'])}/{int(row['total'])})",
                va='center', fontsize=9)
    
    n_total = int(subset['total'].iloc[0]) if len(subset) > 0 else 0
    ax.set_title(f'{result_type} ({n_total} matches)', fontsize=13, fontweight='bold',
                 color=result_colors[result_type])
    ax.set_xlim(0, 1)
    ax.grid(True, alpha=0.3, axis='x')

fig.suptitle('Accuracy models per typ wyniku', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

result_dist = df.groupby(['date', 'home_team', 'away_team', 'actual_result']).ngroups
result_counts = df.drop_duplicates(['date', 'home_team', 'away_team'])['actual_result'].value_counts()
print(f"\nResult distribution: {dict(result_counts)}")
total = result_counts.sum()
for r, c in result_counts.items():
    print(f"  {r}: {c} ({c/total:.1%})")

## 5. Prediction confidence analysis (calibration)

In [ ]:
# Bin confidence and check if higher confidence = better accuracy
bins = [0, 35, 40, 45, 50, 60, 100]
labels = ['<35%', '35-40%', '40-45%', '45-50%', '50-60%', '>60%']
df['conf_bin'] = pd.cut(df['confidence'], bins=bins, labels=labels, right=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1) Calibration - all models combined
calib = df.groupby('conf_bin', observed=True).agg(
    accuracy=('correct', 'mean'),
    count=('correct', 'count'),
).reset_index()

ax = axes[0]
bars = ax.bar(calib['conf_bin'].astype(str), calib['accuracy'], color='steelblue',
              edgecolor='black', alpha=0.85)
for bar, (_, row) in zip(bars, calib.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{row['accuracy']:.0%}\n(n={int(row['count'])})",
            ha='center', fontsize=9)
ax.set_xlabel('Prediction confidence', fontsize=12)
ax.set_ylabel('Actual accuracy', fontsize=12)
ax.set_title('Model calibration (all)', fontsize=13, fontweight='bold')
ax.axhline(y=df['correct'].mean(), color='red', linestyle='--', alpha=0.5, label=f"Average: {df['correct'].mean():.1%}")
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

# 2) Calibration per model (top 4)
ax = axes[1]
top_models = model_stats.head(4)['model'].tolist()
width = 0.18
x = np.arange(len(calib))
for i, model in enumerate(top_models):
    model_calib = df[df['model'] == model].groupby('conf_bin', observed=True)['correct'].mean()
    vals = [model_calib.get(b, 0) for b in labels]
    ax.bar(x + i * width, vals, width, label=model, edgecolor='black', alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(labels)
ax.set_xlabel('Prediction confidence', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Calibration - top 4 models', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Consensus (model agreement) vs accuracy

In [ ]:
matches_df = df.drop_duplicates(['date', 'home_team', 'away_team'])[
    ['date', 'league', 'home_team', 'away_team', 'actual_result',
     'consensus_pred', 'consensus_agreement', 'consensus_correct']
].copy()

# Consensus accuracy per agreement threshold
bins_cons = [0, 50, 62.5, 75, 87.5, 100.1]
labels_cons = ['<50%', '50-62%', '62-75%', '75-87%', '87-100%']
matches_df['agreement_bin'] = pd.cut(matches_df['consensus_agreement'],
                                     bins=bins_cons, labels=labels_cons, right=False)

cons_calib = matches_df.groupby('agreement_bin', observed=True).agg(
    accuracy=('consensus_correct', 'mean'),
    count=('consensus_correct', 'count'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
bars = ax.bar(cons_calib['agreement_bin'].astype(str), cons_calib['accuracy'],
              color='#3498db', edgecolor='black')
for bar, (_, row) in zip(bars, cons_calib.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{row['accuracy']:.0%}\n(n={int(row['count'])})",
            ha='center', fontsize=10)
ax.set_xlabel('Model agreement (%)', fontsize=12)
ax.set_ylabel('Accuracy consensus', fontsize=12)
ax.set_title('Does higher agreement lead to better prediction?', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
ax.hist(matches_df['consensus_agreement'], bins=20, color='#2ecc71',
        edgecolor='black', alpha=0.8)
ax.set_xlabel('Agreement (%)', fontsize=12)
ax.set_ylabel('Liczba matches', fontsize=12)
ax.set_title('Model agreement distribution', fontsize=13, fontweight='bold')
ax.axvline(matches_df['consensus_agreement'].mean(), color='red', linestyle='--',
           label=f"Average: {matches_df['consensus_agreement'].mean():.1f}%")
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

cons_acc = matches_df['consensus_correct'].mean()
print(f"\nConsensus accuracy: {cons_acc:.1%} ({int(matches_df['consensus_correct'].sum())}/{len(matches_df)} matches)")
print(f"Average agreement: {matches_df['consensus_agreement'].mean():.1f}%")

## 7. Confusion matrices per model and summary

In [ ]:
from sklearn.metrics import confusion_matrix as cm_func

show_models = model_stats.head(4)['model'].tolist()
if 'Ensemble' not in show_models:
    show_models.append('Ensemble')

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
class_names = ['HOME', 'DRAW', 'AWAY']

for i, model in enumerate(show_models):
    mdf = df[df['model'] == model]
    cm = cm_func(mdf['actual_result'], mdf['prediction'], labels=class_names)
    acc = mdf['correct'].mean()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=class_names, yticklabels=class_names)
    axes[i].set_title(f'{model}\nAcc: {acc:.1%}', fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Prediction')

cm_cons = cm_func(matches_df['actual_result'], matches_df['consensus_pred'], labels=class_names)
cons_acc = matches_df['consensus_correct'].mean()
sns.heatmap(cm_cons, annot=True, fmt='d', cmap='Greens', ax=axes[len(show_models)],
            xticklabels=class_names, yticklabels=class_names)
axes[len(show_models)].set_title(f'Consensus\nAcc: {cons_acc:.1%}', fontsize=11, fontweight='bold')
axes[len(show_models)].set_ylabel('Actual')
axes[len(show_models)].set_xlabel('Prediction')

for j in range(len(show_models) + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Confusion matrices - rzeczywiste predykcje', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 70)
print("  PREDICTION REPORT ANALYSIS SUMMARY")
print("=" * 70)
print(f"\n  Range:        {min(df['date'])} -> {max(df['date'])}")
print(f"  Days:           {n_dates}")
print(f"  Matches:         {n_matches}")
print(f"  Leagues:          {n_leagues}")
print(f"  Models:        {n_models}")

print(f"\n--- MODEL RANKING ---")
for i, (_, row) in enumerate(model_stats.iterrows(), 1):
    medal = f" {i}."
    print(f"  {medal} {row['model']:25s} -> {row['accuracy']:.1%} "
          f"({int(row['correct'])}/{int(row['total'])} correct, "
          f"avg. confidence: {row['avg_confidence']:.1f}%)")

best_model = model_stats.iloc[0]
worst_model = model_stats.iloc[-1]
spread = best_model['accuracy'] - worst_model['accuracy']

print(f"\n  Best: {best_model['model']} ({best_model['accuracy']:.1%})")
print(f"  Worst: {worst_model['model']} ({worst_model['accuracy']:.1%})")
print(f"  Spread:   {spread:.1%}")
print(f"  Consensus: {cons_acc:.1%}")

print(f"\n--- RESULT DISTRIBUTION ---")
for r in ['HOME', 'DRAW', 'AWAY']:
    c = result_counts.get(r, 0)
    print(f"  {r}: {c} ({c/total:.1%})")

print("\n" + "=" * 70)

## 8. Market accuracy (BTTS, Over/Under, corners, cards)

In [ ]:
market_rows = []
for report in reports:
    report_date = report.get('_date', report.get('date', ''))
    for match in report.get('matches', []):
        if match.get('status') != 'finished':
            continue
        
        home = match.get('home_team', '')
        away = match.get('away_team', '')
        score = match.get('actual_score', '')
        
        markets = match.get('market_predictions', {})
        for market_name, market_data in markets.items():
            actual = market_data.get('actual')
            cons = market_data.get('consensus', {})
            if not cons.get('prediction') or actual is None:
                continue
            
            market_rows.append({
                'date': report_date,
                'match': f"{home} vs {away}",
                'market': market_name,
                'prediction': cons['prediction'],
                'actual': actual,
                'correct': cons['prediction'] == actual,
                'agreement_pct': cons.get('agreement_pct', 0),
            })

market_df = pd.DataFrame(market_rows)

if len(market_df) > 0:
    market_stats = market_df.groupby('market').agg(
        total=('correct', 'count'),
        correct=('correct', 'sum'),
    ).reset_index()
    market_stats['accuracy'] = market_stats['correct'] / market_stats['total']
    market_stats = market_stats.sort_values('accuracy', ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, max(4, len(market_stats) * 0.6)))
    colors = ['#27ae60' if a >= 0.55 else '#e74c3c' if a < 0.45 else '#f39c12' 
              for a in market_stats['accuracy']]
    bars = ax.barh(market_stats['market'], market_stats['accuracy'], color=colors, edgecolor='black')
    for bar, (_, row) in zip(bars, market_stats.iterrows()):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f"{row['accuracy']:.1%} ({int(row['correct'])}/{int(row['total'])})",
                va='center', fontsize=10)
    ax.set_xlabel('Accuracy', fontsize=12)
    ax.set_title('Prediction accuracy per market', fontsize=13, fontweight='bold')
    ax.set_xlim(0, min(1, market_stats['accuracy'].max() * 1.2))
    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
    
    print(f"\nTotal: {int(market_stats['correct'].sum())}/{int(market_stats['total'].sum())} "
          f"({market_stats['correct'].sum()/market_stats['total'].sum():.1%})")
else:
    print("No market data available for analysis.")

## 9. Top Picks accuracy

In [ ]:
def _get_actual_for_market(match_data, market):
    """Get actual result for a market from match data."""
    actual_result = match_data.get('actual_result')
    score = match_data.get('actual_score')
    
    if market == 'result':
        return actual_result
    if market.startswith('double_chance_'):
        if not actual_result:
            return None
        dc_type = market.replace('double_chance_', '').upper()
        hit_map = {
            '1X': actual_result in ['HOME', 'DRAW'],
            'X2': actual_result in ['DRAW', 'AWAY'],
            '12': actual_result in ['HOME', 'AWAY'],
        }
        return dc_type if hit_map.get(dc_type) else f"NOT_{dc_type}"
    
    if market in ('cards_over_3_5', 'cards_over_4_5'):
        total_cards = match_data.get('actual_cards')
        if total_cards is None:
            return None
        threshold = 3.5 if market == 'cards_over_3_5' else 4.5
        return 'OVER' if total_cards > threshold else 'UNDER'
    
    if market in ('corners_over_8_5', 'corners_over_10_5'):
        total_corners = match_data.get('actual_corners')
        if total_corners is None:
            return None
        threshold = 8.5 if market == 'corners_over_8_5' else 10.5
        return 'OVER' if total_corners > threshold else 'UNDER'
    
    if not score:
        return None
    score_str = str(score)
    for sep in [':', '-']:
        if sep in score_str:
            try:
                hg, ag = map(int, score_str.split(sep))
                break
            except (ValueError, TypeError):
                continue
    else:
        return None
    
    total = hg + ag
    if market == 'btts':
        return 'YES' if (hg > 0 and ag > 0) else 'NO'
    elif market == 'over_2_5':
        return 'OVER' if total > 2.5 else 'UNDER'
    elif market == 'over_1_5':
        return 'OVER' if total > 1.5 else 'UNDER'
    return None


tp_results = []
for date_dir in sorted(os.listdir(REPORTS_DIR)):
    date_path = os.path.join(REPORTS_DIR, date_dir)
    if not os.path.isdir(date_path) or len(date_dir) != 10:
        continue
    
    tp_file = os.path.join(date_path, 'top_picks.json')
    if not os.path.exists(tp_file):
        continue
    
    with open(tp_file, 'r', encoding='utf-8') as f:
        tp_data = json.load(f)
    
    report = None
    for r in reports:
        if r.get('_date') == date_dir:
            report = r
            break
    if not report:
        continue
    
    finished = {}
    for m in report.get('matches', []):
        if m.get('status') == 'finished':
            finished[f"{m['home_team']} vs {m['away_team']}"] = m
    
    for match_entry in tp_data.get('picks', []):
        match_name = match_entry['match']
        match_data = finished.get(match_name)
        for bet in match_entry.get('bets', []):
            actual = None
            correct = None
            if match_data:
                actual = _get_actual_for_market(match_data, bet['market'])
                if actual is not None:
                    correct = (bet['pick'] == actual)
            tp_results.append({
                'date': date_dir,
                'match': match_name,
                'market': bet['market'],
                'pick': bet['pick'],
                'score': bet.get('score', 0),
                'actual': actual,
                'correct': correct,
            })

tp_df = pd.DataFrame(tp_results)

if len(tp_df) > 0 and tp_df['correct'].notna().any():
    evaluated = tp_df[tp_df['correct'].notna()]
    
    daily_tp = evaluated.groupby('date').agg(
        hits=('correct', 'sum'),
        total=('correct', 'count'),
    ).reset_index()
    daily_tp['accuracy'] = daily_tp['hits'] / daily_tp['total']
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    ax = axes[0]
    colors = ['#27ae60' if a >= 0.6 else '#e74c3c' if a < 0.4 else '#f39c12' 
              for a in daily_tp['accuracy']]
    bars = ax.bar(range(len(daily_tp)), daily_tp['accuracy'], color=colors, edgecolor='black')
    ax.set_xticks(range(len(daily_tp)))
    ax.set_xticklabels(daily_tp['date'], rotation=45, ha='right', fontsize=9)
    for bar, (_, row) in zip(bars, daily_tp.iterrows()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{int(row['hits'])}/{int(row['total'])}",
                ha='center', fontsize=9)
    ax.set_ylabel('Accuracy')
    ax.set_title('Top Picks - accuracy per day', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
    ax.grid(True, alpha=0.3, axis='y')
    
    ax = axes[1]
    mkt_tp = evaluated.groupby('market').agg(
        hits=('correct', 'sum'),
        total=('correct', 'count'),
    ).reset_index()
    mkt_tp['accuracy'] = mkt_tp['hits'] / mkt_tp['total']
    mkt_tp = mkt_tp.sort_values('accuracy', ascending=True)
    
    colors2 = ['#27ae60' if a >= 0.55 else '#e74c3c' if a < 0.45 else '#f39c12' 
               for a in mkt_tp['accuracy']]
    bars = ax.barh(mkt_tp['market'], mkt_tp['accuracy'], color=colors2, edgecolor='black')
    for bar, (_, row) in zip(bars, mkt_tp.iterrows()):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f"{row['accuracy']:.0%} ({int(row['hits'])}/{int(row['total'])})",
                va='center', fontsize=9)
    ax.set_xlabel('Accuracy')
    ax.set_title('Top Picks - accuracy per market', fontsize=13, fontweight='bold')
    ax.set_xlim(0, min(1, mkt_tp['accuracy'].max() * 1.2))
    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    total_hits = int(evaluated['correct'].sum())
    total_all = len(evaluated)
    print(f"\nTop Picks TOTAL: {total_hits}/{total_all} correct ({100*total_hits/total_all:.1f}%)")
    perfect = (daily_tp['hits'] == daily_tp['total']).sum()
    print(f"Perfect days: {perfect}/{len(daily_tp)}")
else:
    print("No top_picks.json files to analyze (run generate_top_picks.py)")